In [1]:
# MedGraphRAG - Knowledge Graph Builder
# Step 1: Load and explore biomedical data

import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
from datasets import load_dataset

dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", trust_remote_code=True)
df = pd.DataFrame(dataset['train'])
print(df.shape)
print(df.columns.tolist())
df.head(3)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

(1000, 5)
['pubid', 'question', 'context', 'long_answer', 'final_decision']


,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes


In [5]:
# Explore the data
print("Dataset shape:", df.shape)
print("\nFinal decision distribution:")
print(df['final_decision'].value_counts())

print("\nSample question:")
print(df['question'][0])

print("\nSample context (first abstract):")
print(df['context'][0]['contexts'][0][:300])

Dataset shape: (1000, 5)

Final decision distribution:
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64

Sample question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Sample context (first abstract):
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel


In [6]:
# Save data locally
import json

# Flatten context for easier processing
records = []
for _, row in df.iterrows():
    records.append({
        'pubid': row['pubid'],
        'question': row['question'],
        'contexts': row['context']['contexts'],
        'long_answer': row['long_answer'],
        'final_decision': row['final_decision']
    })

with open('../data/pubmedqa.json', 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to data/pubmedqa.json")

Saved 1000 records to data/pubmedqa.json


In [7]:
# Install spaCy biomedical model
import subprocess
subprocess.run(["pip", "install", "scispacy"])
subprocess.run(["pip", "install", "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.4 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'error'


  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [530 lines of output]
      Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
        Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
        Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'error'
        error: subprocess-exited-with-error
      
        × pip subprocess to install build dependencies did not run successfully.
        │ exit code: 1
        ╰─> [500 lines of output]
            Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
              Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
              Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
              Using cached murmurhash-1.0.15-cp313-

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 516.4 kB/s  0:00:28 eta 0:00:010:02:02
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached spacy-3.7.5.tar.gz (1.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'error'


  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [528 lines of output]
      Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
        Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
        Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
        Using cached cymem-2.0.13-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.7 kB)
        Using cached preshed-3.0.13-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.2 kB)
        Using cached murmurhash-1.0.15-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.3 kB)
        Using cached thinc-8.2.5.tar.gz (193 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'error'
        error: subprocess-exited-with-error
      
        × pip subprocess to install build dependencies did not run successfully.
        │ exit code: 1
        ╰─> [498 lines

CompletedProcess(args=['pip', 'install', 'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz'], returncode=1)